In [1]:
"""
Assignment 3 - Question 2.1(c) & 2.1(d): Charging Facility Location
Course: Decision-making in Transport and Mobility (1CM110)
"""

import gurobipy as gp
from gurobipy import GRB
import time
from itertools import combinations


def parse_network(filename):
    """Parse the implicit network file."""
    nodes = set()
    arcs = []
    
    with open(filename, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue
            origin, destination, distance = int(parts[0]), int(parts[1]), int(parts[2])
            nodes.add(origin)
            nodes.add(destination)
            arcs.append((origin, destination, distance))
    
    return nodes, arcs


def parse_od_pairs(filename):
    """Parse the OD pairs file for a company."""
    od_pairs = []
    
    with open(filename, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:
            parts = line.strip().split()
            if len(parts) != 3:
                continue
            od_pairs.append({
                'volume': float(parts[0]),
                'origin': int(parts[1]),
                'destination': int(parts[2])
            })
    
    return od_pairs


def build_arc_sets(arcs, nodes):
    """Build dictionaries for efficient arc lookup."""
    outgoing = {i: [] for i in nodes}
    incoming = {i: [] for i in nodes}
    arc_set = set()
    
    for (i, j, dist) in arcs:
        outgoing[i].append(j)
        incoming[j].append(i)
        arc_set.add((i, j))
    
    return outgoing, incoming, arc_set


def solve_charging_facility_location(nodes, arcs, od_pairs, Q=10, verbose=False):
    """Solve the Capacitated Charging Facility Location Problem."""
    outgoing, incoming, arc_set = build_arc_sets(arcs, nodes)
    K = range(len(od_pairs))
    N = nodes
    
    model = gp.Model("Charging_Facility_Location")
    model.setParam('OutputFlag', 1 if verbose else 0)
    model.setParam('TimeLimit', 600)
    
    y = model.addVars(N, vtype=GRB.BINARY, name="y")
    
    x = {}
    for (i, j, dist) in arcs:
        for k in K:
            x[i, j, k] = model.addVar(lb=0, name=f"x_{i}_{j}_{k}")
    
    model.setObjective(gp.quicksum(y[i] for i in N), GRB.MINIMIZE)
    
    # Flow conservation
    for i in N:
        for k in K:
            if i == od_pairs[k]['origin']:
                d_ik = od_pairs[k]['volume']
            elif i == od_pairs[k]['destination']:
                d_ik = -od_pairs[k]['volume']
            else:
                d_ik = 0
            
            outflow = gp.quicksum(x[i, j, k] for j in outgoing[i] if (i, j, k) in x)
            inflow = gp.quicksum(x[j, i, k] for j in incoming[i] if (j, i, k) in x)
            model.addConstr(outflow - inflow == d_ik, name=f"flow_{i}_{k}")
    
    # Flow requires facility at intermediate nodes
    for (i, j, dist) in arcs:
        for k in K:
            if j != od_pairs[k]['destination']:
                model.addConstr(x[i, j, k] <= y[j], name=f"facility_{i}_{j}_{k}")
    
    # Capacity constraint (exclude flow arriving at destination)
    for i in N:
        total_inflow = gp.quicksum(
            x[j, i, k] 
            for j in incoming[i] 
            for k in K 
            if (j, i, k) in x and i != od_pairs[k]['destination']
        )
        model.addConstr(total_inflow <= Q * y[i], name=f"capacity_{i}")
    
    start_time = time.time()
    model.optimize()
    solve_time = time.time() - start_time
    
    results = {'status': model.status, 'solve_time': solve_time}
    
    if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT:
        results['num_facilities'] = int(model.objVal)
        results['facility_locations'] = sorted([i for i in N if y[i].x > 0.5])
        results['gap'] = model.MIPGap * 100
    else:
        results['num_facilities'] = None
        results['facility_locations'] = None
        results['gap'] = None
    
    return results


def main():
    network_file = "network.txt"
    pairs_files = {'A': "pairsA.txt", 'B': "pairsB.txt", 'C': "pairsC.txt"}
    Q = 10
    companies = ['A', 'B', 'C']
    
    print("Loading network...")
    nodes, arcs = parse_network(network_file)
    print(f"  Nodes: {len(nodes)}, Arcs: {len(arcs)}")
    
    all_od_pairs = {}
    for company, pairs_file in pairs_files.items():
        all_od_pairs[company] = parse_od_pairs(pairs_file)
        print(f"  Company {company} OD pairs: {len(all_od_pairs[company])}")
    
    # Solve for all coalitions (for Shapley value)
    all_results = {}
    
    print()
    print("=" * 70)
    print("Solving for all coalitions")
    print("=" * 70)
    
    for r in range(1, len(companies) + 1):
        for combo in combinations(companies, r):
            combo_name = '+'.join(combo)
            
            combined_od_pairs = []
            for company in combo:
                combined_od_pairs.extend(all_od_pairs[company])
            
            print(f"\n--- Coalition {combo_name} ({len(combined_od_pairs)} OD pairs) ---")
            results = solve_charging_facility_location(nodes, arcs, combined_od_pairs, Q)
            all_results[combo_name] = results
            
            if combo_name == "A+B+C":
                # Save just the node list to a text file
                with open("abc_stations.txt", "w") as f:
                    # Converts [1, 5, 10] into "1,5,10"
                    f.write(",".join(map(str, results['facility_locations'])))
                print("\n>>> SUCCESS: ABC network saved to abc_stations.txt")
                
            if results['num_facilities'] is not None:
                print(f"  Facilities: {results['num_facilities']}, Time: {results['solve_time']:.2f}s")
                print(f"  Locations: {results['facility_locations']}")
            else:
                print(f"  No feasible solution!")
    
    # Summary table
    print()
    print("=" * 70)
    print("SUMMARY TABLE")
    print("=" * 70)
    print(f"{'Coalition':<12} {'# Facilities':<15} {'Time (s)':<12} {'Locations'}")
    print("-" * 70)
    
    for r in range(1, len(companies) + 1):
        for combo in combinations(companies, r):
            combo_name = '+'.join(combo)
            res = all_results[combo_name]
            if res['num_facilities'] is not None:
                print(f"{combo_name:<12} {res['num_facilities']:<15} {res['solve_time']:<12.2f} {res['facility_locations']}")
            else:
                print(f"{combo_name:<12} {'N/A':<15} {res['solve_time']:<12.2f} N/A")
    
    # Shapley value calculation
    def v(coalition):
        if len(coalition) == 0:
            return 0
        key = '+'.join(sorted(coalition))
        return all_results.get(key, {}).get('num_facilities', 0) or 0
    
    v_A, v_B, v_C = v({'A'}), v({'B'}), v({'C'})
    v_AB, v_AC, v_BC = v({'A', 'B'}), v({'A', 'C'}), v({'B', 'C'})
    v_ABC = v({'A', 'B', 'C'})
    
    phi_A = (1/3)*v_A + (1/6)*(v_AB - v_B) + (1/6)*(v_AC - v_C) + (1/3)*(v_ABC - v_BC)
    phi_B = (1/3)*v_B + (1/6)*(v_AB - v_A) + (1/6)*(v_BC - v_C) + (1/3)*(v_ABC - v_AC)
    phi_C = (1/3)*v_C + (1/6)*(v_AC - v_A) + (1/6)*(v_BC - v_B) + (1/3)*(v_ABC - v_AB)
    
    print()
    print("=" * 70)
    print("SHAPLEY VALUE")
    print("=" * 70)
    print(f"{'Company':<12} {'Standalone':<12} {'Shapley':<12} {'Savings'}")
    print("-" * 70)
    print(f"{'A':<12} {v_A:<12} {phi_A:<12.4f} {v_A - phi_A:.4f}")
    print(f"{'B':<12} {v_B:<12} {phi_B:<12.4f} {v_B - phi_B:.4f}")
    print(f"{'C':<12} {v_C:<12} {phi_C:<12.4f} {v_C - phi_C:.4f}")
    print("-" * 70)
    print(f"{'Total':<12} {v_A+v_B+v_C:<12} {phi_A+phi_B+phi_C:<12.4f} (Grand coalition: {v_ABC})")


if __name__ == "__main__":
    main()

Loading network...
  Nodes: 40, Arcs: 374
  Company A OD pairs: 20
  Company B OD pairs: 20
  Company C OD pairs: 20

Solving for all coalitions

--- Coalition A (20 OD pairs) ---
Set parameter Username
Set parameter LicenseID to value 2739960
Academic license - for non-commercial use only - expires 2026-11-17
  Facilities: 8, Time: 5.51s
  Locations: [1, 2, 4, 12, 23, 24, 31, 33]

--- Coalition B (20 OD pairs) ---
  Facilities: 8, Time: 6.67s
  Locations: [2, 5, 12, 15, 17, 22, 30, 33]

--- Coalition C (20 OD pairs) ---
  Facilities: 9, Time: 12.13s
  Locations: [1, 2, 4, 11, 12, 15, 23, 31, 33]

--- Coalition A+B (40 OD pairs) ---
  Facilities: 10, Time: 33.56s
  Locations: [1, 2, 4, 5, 11, 12, 15, 30, 31, 33]

--- Coalition A+C (40 OD pairs) ---
  Facilities: 9, Time: 7.85s
  Locations: [1, 2, 5, 7, 12, 15, 30, 31, 33]

--- Coalition B+C (40 OD pairs) ---
  Facilities: 10, Time: 27.28s
  Locations: [0, 5, 7, 12, 15, 22, 30, 33, 35, 37]

--- Coalition A+B+C (60 OD pairs) ---

>>> SUC

In [3]:
#----------------------------------------------------------
#PART 2.2 WITH 2.1D AS OPENED FACILITIES (coalition ABC)
#----------------------------------------------------------
import heapq
from collections import defaultdict

def build_weighted_adj(arcs):
    """adj[i] = list of (j, dist)"""
    adj = defaultdict(list)
    for i, j, dist in arcs:
        adj[i].append((j, dist))
    return adj
    
# Load network again (need nodes and arcs) (probably not necessary but getting too many errors #@!@#
nodes, arcs = parse_network("network.txt")
adj = build_weighted_adj(arcs)

print("Loaded network:", len(nodes), "nodes,", len(arcs), "arcs")

od_A = parse_od_pairs("pairsA.txt")
od_B = parse_od_pairs("pairsB.txt")
od_C = parse_od_pairs("pairsC.txt")

od_all = od_A + od_B + od_C
print("Total OD pairs:", len(od_all))

# --- Retrieve which facilities are opened as per 2.1d ---
with open("abc_stations.txt", "r") as f:
    data = f.read()
    # Convert the text back into a list of integers
    abc_network = [int(node) for node in data.split(",")]

print(f"{len(abc_network)} stations in the ABC coalition")
print(f"Sstation IDs are: {abc_network}")


Loaded network: 40 nodes, 374 arcs
Total OD pairs: 60
11 stations in the ABC coalition
Sstation IDs are: [1, 2, 4, 5, 7, 12, 15, 22, 31, 33, 35]


In [4]:
#DIJKSTRA ALGORITHM TO FIND SHORTEST PATHS
def dijkstra_path(adj, origin, destination, open_stations):
    """
    Shortest path from origin to destination.
    Constraint: any intermediate node must be in open_stations.
    """
    allowed = set(open_stations) | {origin, destination}

    dist = {origin: 0}
    prev = {}
    pq = [(0, origin)]

    while pq:
        d, u = heapq.heappop(pq)

        if u == destination:
            break
        if d != dist.get(u, float('inf')):
            continue

        for v, w in adj.get(u, []):
            if v not in allowed:
                continue
            nd = d + w
            if nd < dist.get(v, float('inf')):
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))

    if destination not in dist:
        return None, float('inf')

    # reconstruct path
    path = [destination]
    cur = destination
    while cur != origin:
        cur = prev[cur]
        path.append(cur)
    path.reverse()

    return path, dist[destination]


In [5]:
Q = 10
open_stations = abc_network  #Renaming so it’s clearer

total_selfish_time = 0.0
station_load = defaultdict(float)
paths = []  # store per OD if you want

for k, od in enumerate(od_all):
    V = od["volume"]
    O = od["origin"]
    D = od["destination"]

    path, dist = dijkstra_path(adj, O, D, open_stations)

    if path is None:
        raise RuntimeError(f"No feasible selfish path for OD {k}: {O}->{D}")

    total_selfish_time += V * dist

    # Load only at intermediate nodes (exclude origin & destination)
    for node in path[1:-1]:
        if node in open_stations:
            station_load[node] += V

    paths.append((k, V, O, D, path, dist))

print("Selfish total travel time:", total_selfish_time)

violations = {i: station_load[i] - Q for i in station_load if station_load[i] > Q}

print("Numbre of stations used:", len(station_load))
print("Number of violating stations:", len(violations))

if violations:
    for i in sorted(violations):
        print(f"Station {i}: load={station_load[i]:.2f} > Q={Q} (violation={violations[i]:.2f})")
else:
    print("No capacity violations under shellfish routing.")


Selfish total travel time: 28838.142000000003
Numbre of stations used: 11
Number of violating stations: 4
Station 7: load=11.08 > Q=10 (violation=1.08)
Station 12: load=12.17 > Q=10 (violation=2.17)
Station 15: load=12.18 > Q=10 (violation=2.18)
Station 31: load=11.26 > Q=10 (violation=1.26)


In [15]:
import pandas as pd

Q = 10

# --- Station-level table ---
df_stations = pd.DataFrame({
    "station": sorted(open_stations),
    "load_selfish": [station_load.get(i, 0.0) for i in sorted(open_stations)]
})
df_stations["Q"] = Q
df_stations["violation"] = (df_stations["load_selfish"] - Q).clip(lower=0)
df_stations["violating"] = df_stations["violation"] > 0

df_stations


,station,load_selfish,Q,violation,violating
0,1,9.812,10,0.000,False
1,2,2.857,10,0.000,False
2,4,8.313,10,0.000,False
3,5,6.234,10,0.000,False
4,7,11.077,10,1.077,True
5,12,12.172,10,2.172,True
6,15,12.176,10,2.176,True
7,22,6.799,10,0.000,False
8,31,11.262,10,1.262,True
9,33,8.502,10,0.000,False


In [17]:
df_summary = pd.DataFrame([{
    "routing": "Selfish shortest paths",
    "total travel time": total_selfish_time,
    "stations open": len(open_stations),
    "stations violating": int((df_stations["violation"] > 0).sum()),
    "max violation": float(df_stations["violation"].max()),
    "total violation": float(df_stations["violation"].sum())
}])

df_summary


,routing,total travel time,stations open,stations violating,max violation,total violation
0,Selfish shortest paths,28838.142,11,4,2.176,6.687


In [39]:
#FINDING THE SYSTEM OPTIMUM aka ACCOUNT FOR LOAD CAPACITY
def compute_system_optimum_time(nodes, arcs, od_all, open_stations, Q=10, timelimit=120):
    open_set = set(open_stations)
    outgoing, incoming, dist = build_arc_sets(arcs, nodes)

    m = gp.Model("SystemOptimalRouting")
    m.setParam("OutputFlag", 0)
    m.setParam("TimeLimit", timelimit)

    K = range(len(od_all))
    x = {}
    for (i, j, d) in arcs:
        for k in K:
            x[i, j, k] = m.addVar(lb=0.0, name=f"x_{i}_{j}_{k}")

    #Objective: minimiing total travel time
    m.setObjective(
        gp.quicksum(d * x[i, j, k] for (i, j, d) in arcs for k in K),
        GRB.MINIMIZE
    )

    #Flow conservation
    for k in K:
        O, D, V = od_all[k]["origin"], od_all[k]["destination"], od_all[k]["volume"]
        for i in nodes:
            outflow = gp.quicksum(x[i, j, k] for j in outgoing[i])
            inflow  = gp.quicksum(x[j, i, k] for j in incoming[i])
            if i == O:
                m.addConstr(outflow - inflow == V)
            elif i == D:
                m.addConstr(outflow - inflow == -V)
            else:
                m.addConstr(outflow - inflow == 0)

    #Feasibility: intermediate nodes must be open stations (except for destination)
    for k in K:
        D = od_all[k]["destination"]
        for (i, j, d) in arcs:
            if j != D and j not in open_set:
                m.addConstr(x[i, j, k] == 0.0)

    #Capacity  (matches Jie's 2.1d interpretation)
    for i in open_set:
        inflow_to_charge = gp.quicksum(
            x[j, i, k]
            for j in incoming[i]
            for k in K
            if i != od_all[k]["destination"]
        )
        m.addConstr(inflow_to_charge <= Q, name=f"cap_{i}")

    m.optimize()

    res = {"status": m.status, "solve_time": m.Runtime}

    if m.status == GRB.OPTIMAL:
        res["total_travel_time"] = m.objVal

        # Compute station loads (same definition as capacity constraint)
        station_load_SO = {}
        for i in open_set:
            load = sum(
                x[j, i, k].X
                for j in incoming[i]
                for k in K
                if i != od_all[k]["destination"]
            )
            station_load_SO[i] = load

        res["station_load"] = station_load_SO
        res["violations"] = {i: station_load_SO[i] - Q for i in station_load_SO if station_load_SO[i] > Q}

    else:
        res["total_travel_time"] = None
        res["station_load"] = None
        res["violations"] = None
        if m.status == GRB.INFEASIBLE:
            print("Warning: System-optimal model infeasible with these fixed stations + Q.")

    return res

# NAMING THE FUNCTION SYSTEM OPTIMUM SO I CAN RECALL IT LATER
# This runs the Gurobi model and saves the output into 'so_results'
so_results = compute_system_optimum_time(nodes, arcs, od_all, open_stations, Q=10)

#confirmation IT WORKS!!
print(f"Optimization Finished. Status: {so_results['status']}")

Optimization Finished. Status: 2


In [40]:
if so_results["station_load"] is not None:
    print(f"{'Station':<10} {'SO Load':<10} {'Violation':<10}")
    print("-" * 32)
    for i in sorted(so_results["station_load"]):
        load = so_results["station_load"][i]
        viol = max(0.0, load - 10)
        print(f"{i:<10} {load:<10.3f} {viol:<10.3f}")


Station    SO Load    Violation 
--------------------------------
1          9.812      0.000     
2          2.857      0.000     
4          8.313      0.000     
5          6.234      0.000     
7          10.000     0.000     
12         10.000     0.000     
15         10.000     0.000     
22         10.000     0.000     
31         9.143      0.000     
33         9.596      0.000     
35         7.227      0.000     


In [41]:
import pandas as pd

df_compare = pd.DataFrame([
    {
        "Routing": "System optimum (capacity respected)",
        "Total travel time": so_results["total_travel_time"],
        "Violating stations": len(so_results["violations"]) if so_results["violations"] else 0,
        "Total violation": float(sum(so_results["violations"].values())) if so_results["violations"] else 0.0,
        "Max violation": float(max(so_results["violations"].values())) if so_results["violations"] else 0.0,
    },
    {
        "Routing": "Selfish shortest paths (capacity ignored)",
        "Total travel time": total_selfish_time,
        "Violating stations": len(violations),
        "Total violation": float(sum(violations.values())) if violations else 0.0,
        "Max violation": float(max(violations.values())) if violations else 0.0,
    }
])

df_compare


,Routing,Total travel time,Violating stations,Total violation,Max violation
0,System optimum (capacity respected),29075.645,1,1.776357e-15,1.776357e-15
1,Selfish shortest paths (capacity ignored),28838.142,4,6.687000e+00,2.176000e+00
